# Тестирование производительности функций HypEx 

## Генератор синтетических данных

In [1]:
import numpy as np
import pandas as pd

In [15]:
def create_test_data(
    n_columns: int = 1000,
    n_rows: int = 1000,
    n2c_ratio: float = 0.5,
    rs: int = None,
    num_range: tuple = (1, 99),  # Define range for integer values
    n_categories = 5
):
    """Creates data for tests.

    Args:
        n_columns: number of columns
        n_rows: number of rows
        n2c_ratio: ratio of numerical to categorical columns
        rs: random state for np
        num_range: range for integer values (inclusive, exclusive)

    Returns:
        data: DataFrame with the generated test data
    """

    # Set random seed if provided
    if rs is not None:
        np.random.seed(rs)

    # Calculate the number of numerical and categorical columns
    n_numerical = int(n_columns * n2c_ratio)
    n_categorical = n_columns - n_numerical

    # Generate numerical data as integers within the specified range
    numerical_data = np.random.randint(
        num_range[0], num_range[1], size=(n_rows, n_numerical)
    )

    # Generate categorical data
    categories = [f"Category_{i+1}" for i in range(n_categories)]
    categorical_data = np.random.choice(categories, size=(n_rows, n_categorical))

    # Create DataFrame
    data = pd.DataFrame(
        np.hstack((numerical_data, categorical_data)),
        columns=[f"num_col_{i}" for i in range(n_numerical)]
        + [f"cat_col_{i}" for i in range(n_categorical)],
    )

    return data

## Тестирование производительности: замеры времени и памяти

In [3]:
import time
import memory_profiler


def test_function_performance(func, use_time=True, use_memory=True, param_dict=None):
    """
    Tests the performance of the specified function.

    :param func: The function to test
    :param args: Positional arguments for the function
    :param use_time: If True, measures execution time
    :param use_memory: If True, measures memory usage
    :param kwargs: Keyword arguments for the function
    :return: The result of the function execution
    """
    param_dict = param_dict or {}
    exec_time = None
    memory_usage = None

    if use_time:
        start_time = time.time()

    if use_memory:
        start_memory = memory_profiler.memory_usage()[0]

    # Call the function being tested
    result = func(**param_dict)

    if use_memory:
        end_memory = memory_profiler.memory_usage()[0]
        memory_usage = end_memory - start_memory

    if use_time:
        end_time = time.time()
        exec_time = end_time - start_time

    # Output results
    if use_time:
        print(f"Execution time of '{func.__name__}': {exec_time:.4f} seconds")

    if use_memory:
        print(f"Memory usage of '{func.__name__}': {memory_usage:.4f} MiB")

    return result

### AA testing (random split testing)

In [4]:
from hypex.dataset import (
    Dataset,
    InfoRole,
    TreatmentRole,
    TargetRole,
    StratificationRole,
)
from hypex import AATest

In [16]:
data = create_test_data(8, 1000)
data = Dataset(
    roles={
        # data.columns[0]: TreatmentRole(int),
        data.columns[1]: TargetRole(int),
        data.columns[-1]: TargetRole(),
        # cur_data.columns[2]: StratificationRole(str)
    },
    data=data,
)
data

    num_col_0  num_col_1 num_col_2 num_col_3   cat_col_0   cat_col_1  \
0          45         37        27        82  Category_2  Category_5   
1          63         22        35        23  Category_5  Category_3   
2          18         96        49        41  Category_3  Category_5   
3          17         34         3        28  Category_1  Category_2   
4          17         16        41        27  Category_1  Category_4   
..        ...        ...       ...       ...         ...         ...   
995        10         33         4        60  Category_3  Category_1   
996         3         67        39        79  Category_5  Category_2   
997        24         38        36        86  Category_4  Category_1   
998        10         40        19        33  Category_4  Category_4   
999        23         67        87        97  Category_2  Category_4   

      cat_col_2   cat_col_3  
0    Category_1  Category_5  
1    Category_4  Category_5  
2    Category_5  Category_5  
3    Category_5

In [17]:
aa = AATest(n_iterations=10)
res = aa.execute(data)

In [18]:
res.resume

     feature group Chi2Test aa test TTest aa test KSTest aa test  \
0  cat_col_3  test           NOT OK           NaN            NaN   
1  num_col_1  test              NaN        NOT OK         NOT OK   

  TTest best split KSTest best split Chi2Test best split  result  difference  \
0              NaN               NaN                  OK  NOT OK         NaN   
1               OK                OK                 NaN  NOT OK      -0.084   

   difference %  
0           NaN  
1     -0.175453  